# A Hierarchical Model

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. Other examples of the section give nodes without rules to the keyword `fallback = LocalLaplace()`. Here every node has its rules, and the fallback still has a job: one posterior is a product of messages with no closed form, and the fallback approximates it.

The model is the *pump failures* of Gaver and O'Muircheartaigh (1987), a classic of Gibbs sampling (Gelfand and Smith, 1990) and the *Pumps* example of the BUGS manual. Ten pumps of a nuclear power plant have failed $y_i$ times in $t_i$ thousand hours of operation. Each pump has its own failure rate $\lambda_i$, and the rates share a Gamma distribution whose shape $\alpha$ and rate $\beta$ are learned:

$$\begin{aligned}
\alpha &\sim \mathrm{Exponential}(1),\\
\beta &\sim \mathrm{Gamma}(0.1, 1),\\
\lambda_i &\sim \mathrm{Gamma}(\alpha, \beta),\\
y_i &\sim \mathrm{Poisson}(\lambda_i t_i).
\end{aligned}$$

Under the mean-field factorisation $q(\lambda, \alpha, \beta) = q(\lambda)\,q(\alpha)\,q(\beta)$, the rates and $\beta$ are conjugate: their posteriors are Gamma distributions, which RxInfer computes with exact rules. The shape $\alpha$ is not. Its message from each $\lambda_i$ is $\exp(\alpha\,\mathbb{E}[\log \beta \lambda_i] - \log\Gamma(\alpha))$, and the product of ten of them with the prior is a distribution of no known family. Without the fallback, RxInfer asks for a form constraint on $\alpha$, chosen by hand. With it, `LocalLaplace()` projects the product onto a Gaussian in $\log \alpha$ by the Laplace method, and `NaturalGradientPropagation()` gives each of the ten messages a site in the family of $\alpha$'s prior, the Exponential, a Gamma of shape 1: the posterior of $\alpha$ stays a Gamma distribution.

In [ ]:
using RxInfer, BayesBase, Distributions, SpecialFunctions
using Plots, PrettyTables, Printf

## Data

In [ ]:
t = [94.3, 15.7, 62.9, 126.0, 5.24, 31.4, 1.05, 1.05, 2.1, 10.5] # thousands of hours
y = [5, 1, 5, 14, 3, 19, 1, 1, 4, 22]                           # failures

bar(y ./ t; xlabel = "pump", ylabel = "failures per thousand hours", label = "yᵢ / tᵢ", color = :gray)

## The model

`Exponential` is not a factor node in RxInfer, so the prior on $\alpha$ is written as the Gamma distribution it is, with shape 1 and rate 1. The rate of a Poisson count is `λ[i] * t[i]`, the product of a variable and a constant, for which RxInfer has rules.

In [ ]:
@model function pumps(y, t)
    α ~ Gamma(shape = 1.0, rate = 1.0)
    β ~ Gamma(shape = 0.1, rate = 1.0)
    for i in eachindex(y)
        λ[i] ~ Gamma(shape = α, rate = β)
        y[i] ~ Poisson(λ[i] * t[i])
    end
end

mean_field = @constraints begin
    q(λ, α, β) = q(λ)q(α)q(β)
end

initial = @initialization begin
    q(α) = GammaShapeRate(1.0, 1.0)
    q(β) = GammaShapeRate(1.0, 1.0)
    q(λ) = GammaShapeRate(1.0, 1.0)
end

## Inference

We run both schemes for 100 iterations of variational message passing, with the free energy.

Under `NaturalGradientPropagation()`, each rate's message towards $\alpha$, a log-density $\ell_i(\alpha) = \alpha\,\mathbb{E}[\log \beta\lambda_i] - \log\Gamma(\alpha)$, has a Gamma site on its edge, and $q(\alpha)$ is the prior times the ten sites. At each iteration a site takes a damped heavy-ball step in the Gamma's natural parameters, $v \leftarrow b\,v + a(\eta^\star - \eta)$ and $\eta \leftarrow \eta + v$, with the step size $a$ = `alpha` = 0.5 and the momentum $b$ = `beta` = 0.2 by default, keywords of the scheme unrelated to the model's $\alpha$ and $\beta$, towards its target $\eta^\star = G(q)^{-1}\,\mathrm{Cov}_q[T(\alpha), \ell_i(\alpha)]$: the natural gradient of $\mathbb{E}_q[\ell_i]$, with $T(\alpha) = (\log\alpha, \alpha)$ the Gamma's sufficient statistics and $G$ its Fisher information, by quadrature in $\log\alpha$. At the fixed point every site is its target, and $q(\alpha)$ is the best Gamma under mean field: the stationary point of the free energy over Gammas, as conjugate-computation variational inference would reach in that family.

In [ ]:
fit(fallback) = infer(
    model = pumps(t = t), data = (y = y,),
    constraints = mean_field, initialization = initial, iterations = 100,
    fallback = fallback, free_energy = true, report = true,
)

laplace = fit(LocalLaplace())
ngmp = fit(NaturalGradientPropagation())

qα_laplace, qα_ngmp = laplace.posteriors[:α][end], ngmp.posteriors[:α][end]
qβ_laplace, qβ_ngmp = laplace.posteriors[:β][end], ngmp.posteriors[:β][end]

## What was approximated

The report of the Laplace run says what the keyword did. Only $\alpha$'s posterior was approximated: the rates and $\beta$ are exact Gamma distributions. No message needed the fallback, because every node has its rules.

The free energy needs one thing more. The product node `λ[i] * t[i]` has message rules but no rule for the joint of its inputs, which the free energy scores. The fallback gives it: with $t_i$ known, that joint is $\lambda_i$'s own marginal, the product of its message and the node's message towards it, in closed form.

In [ ]:
records = laplace.report.approximations
@assert count(RxInfer.is_approximated, values(records.variables)) == 1 # hide
@assert RxInfer.is_approximated(records.variables[(:α, nothing)]) # hide
@assert all(r -> r.fallbacks == 0, (r for ((node, kind, target), r) in RxInfer.node_types(records) if kind === :message)) # hide
@assert isempty(RxInfer.diagnostics(laplace.report)) && isempty(RxInfer.diagnostics(ngmp.report)) # hide
@assert ngmp.report.approximations.variables[(:α, nothing)].sites.outcome.method === :natural_gradient # hide
records

Laplace's Gaussian on $\log \alpha$ is a log-normal distribution on $\alpha$. Natural-gradient propagation's posterior of $\alpha$ is a Gamma distribution, the prior times ten Gamma sites; its report counts the sites and their steps, and says that every site has settled:

In [ ]:
@assert qα_ngmp isa GammaShapeRate # hide
qα_laplace.gaussian, qα_ngmp

In [ ]:
ngmp.report.approximations

## The exact posterior

The rates integrate out in closed form: a Poisson count whose rate has a Gamma distribution is negative binomial,

$$p(y_i \mid \alpha, \beta) = \frac{\Gamma(y_i + \alpha)}{\Gamma(\alpha)\, y_i!} \left(\frac{\beta}{\beta + t_i}\right)^{\alpha} \left(\frac{t_i}{\beta + t_i}\right)^{y_i}.$$

The posterior of $(\alpha, \beta)$ is then two-dimensional, and a grid in log coordinates gives it, with the evidence $p(y)$.

In [ ]:
lognegbin(α, β) = sum(loggamma(y[i] + α) - loggamma(α) - loggamma(y[i] + 1) + α * log(β / (β + t[i])) + y[i] * log(t[i] / (β + t[i])) for i in eachindex(y))
logjoint(α, β) = logpdf(Gamma(1.0, 1.0), α) + logpdf(Gamma(0.1, 1.0), β) + lognegbin(α, β)

us, vs = range(log(0.02), log(10), length = 1200), range(log(0.01), log(20), length = 1200)
grid = [logjoint(exp(u), exp(v)) + u + v for u in us, v in vs] # the Jacobian of the log coordinates
top = maximum(grid)
weights = exp.(grid .- top)
logevidence = top + log(sum(weights) * step(us) * step(vs))
weights ./= sum(weights)

αgrid, βgrid = exp.(us), exp.(vs)
Eα = sum(weights .* αgrid)
Eβ = sum(weights .* βgrid')
sdα = sqrt(sum(weights .* (αgrid .- Eα) .^ 2))
sdβ = sqrt(sum(weights .* (βgrid' .- Eβ) .^ 2))
@printf("exact: E[α] = %.3f (sd %.3f), E[β] = %.3f (sd %.3f), -log p(y) = %.3f\n", Eα, sdα, Eβ, sdβ, -logevidence)
@assert isapprox(Eα, 0.70; atol = 0.01) && isapprox(Eβ, 0.93; atol = 0.01) # hide

The means agree with the Gibbs sampler of the BUGS manual, which reports about 0.70 and 0.92.

## The best that mean field can do

The two schemes approximate twice: by the factorisation, and by the Gaussian on $\log\alpha$. To tell the two apart, we compute the best mean-field posterior, with $q(\alpha)$ of any shape, on a fine grid. Coordinate ascent alternates the three exact updates, of which only $\alpha$'s needs the grid:

$$q(\alpha) \propto p(\alpha) \exp\Big(\sum_i \alpha\,\mathbb{E}[\log\beta] - \log\Gamma(\alpha) + (\alpha - 1)\,\mathbb{E}[\log\lambda_i]\Big).$$

In [ ]:
function best_mean_field(; iterations = 500)
    as = range(1.0e-4, 4.0, length = 40_000)
    Eα, Elogβ, Eβ = 1.0, 0.0, 1.0
    qλ, qβ, w = nothing, nothing, nothing
    for _ in 1:iterations
        qλ = [Gamma(Eα + y[i], 1 / (Eβ + t[i])) for i in eachindex(y)]
        Elogλ = [digamma(shape(q)) + log(scale(q)) for q in qλ]
        qβ = Gamma(0.1 + length(y) * Eα, 1 / (1 + sum(mean, qλ)))
        Elogβ, Eβ = digamma(shape(qβ)) + log(scale(qβ)), mean(qβ)
        logw = [-a + sum(a * Elogβ - loggamma(a) + (a - 1) * Elogλ[i] for i in eachindex(y)) for a in as]
        w = exp.(logw .- maximum(logw))
        w ./= sum(w)
        Eα = sum(w .* as)
    end
    # its free energy: the expected negative log joint, minus the entropies
    Elogλ = [digamma(shape(q)) + log(scale(q)) for q in qλ]
    U = Eα - (-0.9 * Elogβ - Eβ - loggamma(0.1))
    for i in eachindex(y)
        U -= Eα * Elogβ - sum(w .* loggamma.(as)) + (Eα - 1) * Elogλ[i] - Eβ * mean(qλ[i])
        U -= y[i] * (log(t[i]) + Elogλ[i]) - t[i] * mean(qλ[i]) - loggamma(y[i] + 1)
    end
    H = -sum(p * log(p / step(as)) for p in w if p > 0) + entropy(qβ) + sum(entropy, qλ)
    return (; α = as, density = w ./ step(as), mean = Eα, sd = sqrt(sum(w .* (as .- Eα) .^ 2)), β = qβ, free_energy = U - H)
end

best = best_mean_field()

## Results

One correction comes first. RxInfer's free energy takes each rate's term from the Gamma node's average energy, which needs $\mathbb{E}[\log\Gamma(\alpha)]$ under $q(\alpha)$. For a Gamma $q(\alpha)$, the node's rule takes it from ExponentialFamily.jl's closed form, built on Stirling's approximation of $\log\Gamma$, which is accurate for a large $\alpha$ but too low at $\alpha$ near 0.65, for each of the ten rates. We add the difference back, with $\mathbb{E}[\log\Gamma(\alpha)]$ by quadrature. Laplace's $q(\alpha)$ is not a Gamma, and its expectations are by quadrature already.

In [ ]:
# E[log Γ(α)] under the Gamma posterior of α, by quadrature, against the closed form the free energy took
grid_α = range(1.0e-6, 5.0, length = 100_000)
density_α = pdf.(Gamma(shape(qα_ngmp), 1 / rate(qα_ngmp)), grid_α)
E_loggamma = sum(density_α .* loggamma.(grid_α)) / sum(density_α)
correction = length(y) * (E_loggamma - mean(loggamma, qα_ngmp))
F_ngmp = last(ngmp.free_energy) + correction
@printf("E[log Γ(α)]: quadrature %.4f, closed form %.4f; free energy %.4f + %.4f = %.4f\n", E_loggamma, mean(loggamma, qα_ngmp), last(ngmp.free_energy), correction, F_ngmp)

rows = [
    "exact" Eα sdα Eβ sdβ -logevidence;
    "best mean field" best.mean best.sd mean(best.β) std(best.β) best.free_energy;
    "NaturalGradientPropagation()" mean(qα_ngmp) std(qα_ngmp) mean(qβ_ngmp) std(qβ_ngmp) F_ngmp;
    "LocalLaplace()" mean(qα_laplace) std(qα_laplace) mean(qβ_laplace) std(qβ_laplace) last(laplace.free_energy)
]
@assert abs(mean(qα_ngmp) - best.mean) < 0.002 && abs(std(qα_ngmp) - best.sd) < 0.002 # hide
@assert abs(mean(qβ_ngmp) - mean(best.β)) < 0.002 # hide
@assert abs(F_ngmp - best.free_energy) < 0.005 # hide
@assert -logevidence < best.free_energy < last(laplace.free_energy) && F_ngmp < last(laplace.free_energy) # hide
@assert abs(mean(qα_laplace) - Eα) < abs(mean(qα_ngmp) - Eα) # hide
@assert max(std(qα_ngmp), std(qα_laplace)) < 0.75 * sdα && max(std(qβ_ngmp), std(qβ_laplace)) < 0.65 * sdβ # hide
pretty_table(rows; column_labels = ["", "E[α]", "sd(α)", "E[β]", "sd(β)", "free energy"], formatters = [fmt__printf("%.3f", 2:6)], fit_table_in_display_horizontally = false)

The last column is the free energy, an upper bound on $-\log p(y)$ under mean field, which the first row gives.

In [ ]:
αs = range(0.05, 2.0, length = 400)
exact_α = vec(sum(weights; dims = 2)) ./ (αgrid .* step(us)) # the marginal of α, in α's own coordinates
lognormal(q) = LogNormal(mean(q.gaussian), std(q.gaussian))

plot(αgrid, exact_α; xlims = (0, 2), color = :black, linewidth = 2, label = "exact", xlabel = "α", ylabel = "density")
plot!(best.α, best.density; color = :gray, linestyle = :dash, linewidth = 2, label = "best mean field")
plot!(αs, pdf.(qα_ngmp, αs); color = :red, label = "NaturalGradientPropagation()")
plot!(αs, pdf.(lognormal(qα_laplace), αs); color = :blue, label = "LocalLaplace()")

Three things follow.

1. **`NaturalGradientPropagation()` finds the best mean-field posterior.** Its $\alpha$ agrees with the best mean field to within 0.001 in mean and spread, and its free energy is within 0.001 of the best one. The best $q(\alpha)$, of any shape, is close to a Gamma, the family of the prior that the sites keep, so restricting it to that family costs almost nothing here.
2. **Mean field itself misses the spread.** Both schemes, and the best mean field, are overconfident: the standard deviation of $\alpha$ is 0.17 to 0.19 against the exact 0.27, and that of $\beta$ 0.33 to 0.34 against 0.54. Under mean field, $\alpha$ and $\beta$ cannot share what the data say about them together: a larger shape with a larger rate explains the same rates.
3. **Laplace's closer mean is a coincidence.** Laplace's mean of $\alpha$ is closer to the exact one than natural-gradient propagation's, but it is 0.07 above the best mean-field mean, while the factorisation puts that mean 0.045 below the exact one: two errors in opposite directions. Its free energy, the measure that mean field minimises, is the highest of the three.

The free energy ranks the approximations without the exact answer: lower is better, and `NaturalGradientPropagation()`'s is lower than `LocalLaplace()`'s. What it cannot show is what the factorisation costs; only a less factorised model can.

## References

- D. P. Gaver and I. G. O'Muircheartaigh (1987). Robust empirical Bayes analyses of event rates. *Technometrics*, 29(1), 1–15.
- A. E. Gelfand and A. F. M. Smith (1990). Sampling-based approaches to calculating marginal densities. *Journal of the American Statistical Association*, 85(410), 398–409.
- M. Lukashchuk, K. Yemets, A. Ledbetter and İ. Şenöz (2026). Information geometry of message passing. arXiv:2608.15922.
- D. Spiegelhalter, A. Thomas, N. Best and W. Gilks (1996). *BUGS 0.5 Examples, Volume 1*: Pumps. MRC Biostatistics Unit, Cambridge.